In [ ]:
from datetime import datetime as dt

import pandas as pd
import numpy as np
# import matplotlib.pyplot as plt
from mlxtend.frequent_patterns import apriori, association_rules
# import seaborn as sns

from ucimlrepo import fetch_ucirepo 

## 1. データの読み込み

In [ ]:
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 
df_raw = online_retail.data.original

## 2. 前処理

In [ ]:
df_data = df_raw.copy()

In [ ]:
df_data.isnull().sum()

### 2.1 InvoiceNoのキャセル分を削除

In [ ]:
def filter_cancelled(df):

    # 堅牢性のため、大文字として扱う
    # 元データがイギリスのデータであり、日本語ではないため、全角半角変換は入れない
    df["cancelled_flag"] = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")

    return df[~df["cancelled_flag"]].reset_index(drop=True)

### 2.2 InvoiceDateをdatetime型に変換


In [ ]:
def convet_data(df):

    date_format = "%m/%d/%Y %H:%M"

    df["InvoiceDate"] = pd.to_datetime(
        df["InvoiceDate"],
        format=date_format
    )

    return df.reset_index(drop=True)

### 2.3 StockCodeのクレンジング

##### 2.3.1 StockCodeの中身の確認

In [ ]:
np.sort(df_data["StockCode"].unique())

# 公式サイトの定義：a 5-digit integral number uniquely assigned to each distinct product
# しかし、数字５つ＋アルファベットなどのバリエーションが存在する。
# また、POSTでPOSTAGEなど、送料など商品以外を示すstockcodeが存在する

In [ ]:
for i in df_data["StockCode"].unique():
    
    if "37444" in i:
        print(i)
    # if len(i) != 5:
    #     print(i)

    

In [ ]:
df_data[df_data["StockCode"]=="POST"].head(3)

In [ ]:
df_data[df_data["StockCode"]=="m"].head(3)

In [ ]:
query_char = "gift"

df_data[df_data["StockCode"].str.contains(query_char, na=False)].head(5)


##### 2.3.2 StockCodeのうち数字5桁または数字5桁+アルファベット1文字だけを抽出

In [ ]:
def filter_stockcode(df):

    list_filterd_code = []

    list_ucode = df["StockCode"].unique()

    for code in list_ucode:

        # 定義通りの数字5桁だけをとりだす
        if code.isdecimal():
            list_filterd_code.append(code)

        # 数字5桁＋suffixのアルファベット1文字を取り出す
        elif len(code) == 6:
            if code[:5].isdecimal():
                list_filterd_code.append(code)

    # リストの文字を正規表現のOR条件（|）で連結
    pattern = "|".join(list_filterd_code)

    # で条件に一致する行を抽出
    df_filtered = df[df["StockCode"].isin(list_filterd_code)]

    return df_filtered


In [ ]:
df_tmp = filter_stockcode(df=df_data)

In [ ]:
df_tmp.head(10)

### 2.4 前処理の実施

In [ ]:
def preprocessing(df):
    
    df = filter_cancelled(df=df)
    df = convet_data(df=df)
    df = filter_stockcode(df=df)

    return df

In [ ]:
df_filtered = preprocessing(df=df_data)

In [ ]:
df_filtered.head(3)

In [ ]:
df_filtered.shape

In [ ]:
# stockcodeとdescriptionを紐づける
dict_code_to_name = df_filtered.set_index("StockCode")["Description"].to_dict()

## 3. マーケットバスケット分析

### 3.1 同時購入商品の組み合わせの抽出

In [ ]:
# バスケットの確認
basket_tmp = pd.crosstab(df_filtered["InvoiceNo"], df_filtered["StockCode"]).astype(bool)

In [ ]:
basket_tmp[basket_tmp["10002"]==True]

# そのままだとバスケット分析をやるにはデータ量が多い
# 季節ごとに分割して分析する

In [ ]:
# 季節分割ように月を求める
df_filtered["month"] = df_filtered["InvoiceDate"].dt.month

In [ ]:
def mba_analysis(month_duration, df):

    df_term = df[df["month"].isin(month_duration)]

    basket = pd.crosstab(df_term["InvoiceNo"], df_term["StockCode"]).astype(bool)

    frequent_itemsets = apriori(basket, min_support=0.01, use_colnames=True)
    rules = association_rules(frequent_itemsets, metric="lift", min_threshold=1.2)

    strong_rules = rules[rules["confidence"] >= 0.3]

    return strong_rules

In [ ]:
# month_duration = [12,1,2]
# month_duration = [3,4,5]
# month_duration = [6,7,8]
month_duration = [9,10,11]

df_rules_stockcode = mba_analysis(month_duration=month_duration, df=df_filtered)

In [ ]:
def convert_frozenset_to_description(fset, dict_id_to_name):
    # frozensetの中身をリストに変換し、辞書を使って商品名に置換
    names = [str(dict_id_to_name.get(item, item)) for item in fset]
    # 複数ある場合はカンマ区切りの文字列にする（1個ならそのまま商品名になる）
    return ", ".join(names)

In [ ]:
df_rules_description = df_rules_stockcode.copy()

In [ ]:
# stockcodeを商品名に変換する
df_rules_description["antecedents"] = df_rules_description["antecedents"].apply(lambda x: convert_frozenset_to_description(x, dict_code_to_name))
df_rules_description["consequents"] = df_rules_description["consequents"].apply(lambda x: convert_frozenset_to_description(x, dict_code_to_name))

In [ ]:
df_rules_description.head(5)

In [ ]:
df_rules_description.sort_values(["confidence","lift"], ascending=False)

### 3.2 組み合わせの分析

In [ ]:
# 色違いや柄違いの同時購入を弾くために、
# antecedentsとconsequentsの商品名の単語の重複度（Jaccard係数）を計算する
# 簡易的なため、antecedentsが複数商品を含む場合の場合分けはしていない

def calculate_word_jaccard(name_a, name_b):
  
    # 欠損値（nan）の対策と大文字統一
    set_a = set(str(name_a).upper().split())
    set_b = set(str(name_b).upper().split())
    
    # どちらかが空の場合は重複度0にする
    if not set_a or not set_b:
        return 0.0
        
    # 積集合（共通の単語）と和集合（すべての単語）
    intersection = set_a.intersection(set_b)
    union = set_a.union(set_b)
    
    # Jaccard係数 = 共通の単語数 / 全単語数
    return len(intersection) / len(union)

In [ ]:
df_rules_description["word_similarity"] = df_rules_description.apply(
    lambda row: calculate_word_jaccard(row["antecedents"], row["consequents"]), 
    axis=1
)

In [ ]:
threshold = 0.2
df_rules_filtered = df_rules_description[df_rules_description["word_similarity"] < threshold]

In [ ]:
df_rules_filtered.sort_values(["confidence","lift"], ascending=False)

In [ ]:
# 結果の保存
df_rules_filtered.to_csv("data/filtered_rules_9to11.csv", index=False, encoding="utf-8")